# Reusable Customer Churn Prediction Pipeline

This notebook builds a complete scikit-learn pipeline for churn prediction, including data preparation, model training, evaluation, and reuse for new customer data.

## Business Goal
This pipeline predicts which customers are likely to churn so the business can take early retention actions such as discounts, support outreach, or personalized offers.

## Project Summary
- Loads customer churn data from CSV
- Removes the identifier column
- Splits the data into train and test sets
- Builds a reusable scikit-learn pipeline with preprocessing and model training
- Evaluates the classifier and saves the trained pipeline for future use

In [2]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

DATA_PATH = Path('customer_churn.csv')
PIPELINE_PATH = Path('customer_churn_pipeline.pkl')

print('Loading dataset from', DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} does not exist. Please add the customer churn CSV file to this folder.')

print('Dataset loaded successfully')
df = pd.read_csv(DATA_PATH)
print(df.head())
print('\nDataset shape:', df.shape)
print('\nColumn names:', list(df.columns))
print('\nMissing values:\n', df.isnull().sum())
print('\nTarget distribution:\n', df['Churn'].value_counts(dropna=False))

if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

X = df.drop(columns=['Churn'])
y = df['Churn']

numeric_features = ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'SupportTickets']
categorical_features = ['Gender', 'ContractType', 'PaymentMethod', 'InternetService']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print('\nAccuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

joblib.dump(model_pipeline, PIPELINE_PATH)
print('\nPipeline saved to', PIPELINE_PATH)

loaded_pipeline = joblib.load(PIPELINE_PATH)
new_customer = pd.DataFrame({
    'Gender': ['Female'],
    'Age': [32],
    'Tenure': [5],
    'MonthlyCharges': [850],
    'TotalCharges': [4250],
    'ContractType': ['Monthly'],
    'PaymentMethod': ['Credit Card'],
    'InternetService': ['Fiber'],
    'SupportTickets': [3]
})
prediction = loaded_pipeline.predict(new_customer)
print('\nNew customer prediction:', prediction[0])

Loading dataset from customer_churn.csv
Dataset loaded successfully
  CustomerID  Gender  Age  Tenure  MonthlyCharges  TotalCharges  \
0       C001  Female   30      12            65.0           780   
1       C002    Male   45      24            85.0          2040   
2       C003  Female   29       6             NaN           570   
3       C004    Male   37      18            70.0          1260   
4       C005  Female   41       8            90.0           720   

     ContractType     PaymentMethod InternetService  SupportTickets Churn  
0  Month-to-month       Credit card           Fiber             1.0    No  
1        One year     Bank transfer             DSL             0.0    No  
2  Month-to-month  Electronic check           Fiber             4.0   Yes  
3        Two year       Credit card             DSL             1.0    No  
4  Month-to-month  Electronic check           Fiber             5.0   Yes  

Dataset shape: (20, 11)

Column names: ['CustomerID', 'Gender', 'Age', '